# Коробочка другого цвета: маска HSV на C++ (Colab)
Порядок: **запускайте ячейки сверху вниз** (Shift+Enter).
1. Установка OpenCV для C++ (1–2 минуты, один раз)
2. Снимок с камеры (возьмите коробочку **не синего цвета**)
3. Код на C++
4. Компиляция
5. Подбор маски ползунками

## 1. Установка OpenCV для C++

In [ ]:
!apt-get -qq update && apt-get -qq install -y libopencv-dev > /dev/null && echo 'OpenCV установлен'

## 2. Снимок с камеры
Разрешите доступ к камере в браузере, возьмите коробочку в руку и нажмите **Capture**.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.9):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    with open(filename, 'wb') as f:
        f.write(b64decode(data.split(',')[1]))
    return filename

take_photo()
print('Снимок сохранён: photo.jpg')

## 3. Код на C++ (сохраняется в файл box.cpp)

In [ ]:
%%writefile box.cpp
// Поиск коробочки по цвету (маска HSV).
// Запуск: ./box photo.jpg hMin hMax sMin sMax vMin vMax
// Результат: result.png (рамка вокруг коробочки) и mask.png (маска)
#include <opencv2/opencv.hpp>
#include <cstdlib>
#include <iostream>
#include <vector>

int main(int argc, char** argv) {
    if (argc < 2) { std::cerr << "Нужен файл: ./box photo.jpg [hMin hMax sMin sMax vMin vMax]" << std::endl; return 1; }
    cv::Mat img = cv::imread(argv[1]);
    if (img.empty()) { std::cerr << "Не удалось открыть " << argv[1] << std::endl; return 1; }

    // границы маски (по умолчанию - светлая/белая коробка); можно задать аргументами. H: 0..179, S и V: 0..255
    int v[6] = {0, 179, 0, 45, 180, 255};
    for (int i = 0; i < 6 && i + 2 < argc; ++i) v[i] = std::atoi(argv[i + 2]);

    // маска: белым выделены точки, цвет которых попал в диапазон HSV
    cv::Mat hsv, mask;
    cv::cvtColor(img, hsv, cv::COLOR_BGR2HSV);
    cv::inRange(hsv, cv::Scalar(v[0], v[2], v[4]), cv::Scalar(v[1], v[3], v[5]), mask);

    // находим контуры белых областей и рисуем рамку вокруг крупных (мелкий шум пропускаем)
    std::vector<std::vector<cv::Point>> contours;
    cv::findContours(mask.clone(), contours, cv::RETR_EXTERNAL, cv::CHAIN_APPROX_SIMPLE);
    cv::Mat result = img.clone();
    int found = 0;
    for (const auto& c : contours) {
        if (cv::contourArea(c) < 5000) continue;
        cv::rectangle(result, cv::boundingRect(c), cv::Scalar(0, 255, 0), 3);
        ++found;
    }

    cv::imwrite("result.png", result);
    cv::imwrite("mask.png", mask);
    std::cout << "H " << v[0] << "-" << v[1] << "  S " << v[2] << "-" << v[3]
              << "  V " << v[4] << "-" << v[5] << "  | найдено коробок: " << found << std::endl;
    return 0;
}


## 4. Компиляция

In [ ]:
!g++ box.cpp -o box $(pkg-config --cflags --libs opencv4) && echo 'Скомпилировано'

## 5. Подбор маски
Двигайте ползунки, пока справа (маска) не останется **только коробочка**, а слева она не окажется в зелёной рамке. Значения H, S, V печатаются над картинкой.

**Важно:** у каждой пары `min` должен быть **меньше** `max`, иначе маска будет пустой (чёрной).

H: 0–179, S и V: 0–255. Примерно: красный 0–10, оранжевый 10–25, жёлтый 20–35, зелёный 35–85, фиолетовый 130–160. Белая/светлая коробка: S 0–45, V 180–255 (H любой, 0–179).

In [ ]:
import subprocess, cv2
from ipywidgets import interact, IntSlider
from google.colab.patches import cv2_imshow

def run(h_min=0, h_max=179, s_min=0, s_max=45, v_min=180, v_max=255):
    args = [str(x) for x in (h_min, h_max, s_min, s_max, v_min, v_max)]
    out = subprocess.run(['./box', 'photo.jpg'] + args, capture_output=True, text=True)
    print(out.stdout or out.stderr)
    result = cv2.imread('result.png')
    mask = cv2.cvtColor(cv2.imread('mask.png', 0), cv2.COLOR_GRAY2BGR)
    cv2_imshow(cv2.hconcat([result, mask]))      # слева - рамка, справа - маска

def sl(value, mx):      # ползунок с начальным значением; обновляется, когда отпустили
    return IntSlider(value=value, min=0, max=mx, continuous_update=False)

interact(run, h_min=sl(0, 179), h_max=sl(179, 179), s_min=sl(0, 255),
         s_max=sl(45, 255), v_min=sl(180, 255), v_max=sl(255, 255))

Готово: сделайте скриншот этой ячейки (вы с коробочкой в рамке + маска) и приложите вместе с кодом на C++ (ячейка 3).